# 训练 — MSN PCT 颅骨补全

**这个 notebook 只做一件事：发起一次训练，并检查这一次跑得对不对。**
判读结果、跨 run 对比，全部在 [`MSN_compare_runs.ipynb`](MSN_compare_runs.ipynb)。

| 节 | 内容 | 每次要动吗 |
|---|---|---|
| **1. 控制面板** | 这一轮叫什么、用什么配置 | ✅ **只改这里** |
| 2. 预检 | 数据、显存、run 名是否已被占用 | 不用改，跑一下 |
| 3. 训练 | 发起训练，输出直接打在 cell 里 | 不用改 |
| 4. 本轮自检 | 这一次跑得对不对（不是"结果好不好"） | 不用改 |
| 5. 存档 | `run.json` + `history.csv` 进 `experiments_log/` | 不用改 |
| 附录 A | 数据准备（只在改点数/样本量时用） | 基本不用 |
| 附录 B | 本实现相对原 demo 改了什么 | 只读 |

---

## ⚠️ 三条硬规则

1. **kernel 选 `comp0190-msn`**（右上角 Select Kernel）。
2. **本 notebook 不要建模型。** 训练子进程需要 15.5 GiB / 24 GiB，本 kernel 一旦建了模型就占住显存，训练会 OOM。
   看可视化去 compare notebook，那边跑完记得 Restart Kernel 再回来训练。
3. **想跑得更稳就用终端**（断连不中断）。第 1 节会把完整命令打印出来，直接复制：
   ```bash
   cd /root/comp0190-organ-completion && nohup <命令> > train.log 2>&1 &
   tail -f train.log
   ```

## 1. 控制面板 ⬅️ 每次只改这里

**两种模式：**

| 想干什么 | 怎么填 |
|---|---|
| **重复一次已有的 run**（做重复实验） | `FROM_RUN = "tie_qk"`，`RUN_NAME = "tie_qk_r2"`，`EXTRA_FLAGS = []` |
| **试一个新配置** | `FROM_RUN = ""`，自己写 `EXTRA_FLAGS` |
| 在已有配置上只改一项 | `FROM_RUN = "cd_rep05_full"` + `EXTRA_FLAGS = ["--dcd-lambda", "2"]`（会打印警告：这不再是严格重复） |

`FROM_RUN` 会把那个 run 的 `run.json` 里记的**每一个超参**照抄过来（实测 18 个字段），
包括 `--epochs` / `--minutes` 这种容易忘的。**重复实验必须用它** —— 手抄 8 个 flag 正是
重复实验悄悄变成"另一个实验"的地方（`notext` 和对照就只差 9 个 flag 里的 1 个）。

> ⚠️ `RUN_NAME` 撞名会被**拒绝启动**（2026-08-24 加的保护）。`cd_only` 的权重就是被一次
> 同名重跑覆盖掉的，而 `run.json` 只在训练结束时写，所以记录和权重会指向两次不同的训练且不报错。

In [ ]:
import os, sys, json, subprocess, shutil
import numpy as np

# ============================ 每次改这里 ============================
RUN_NAME    = "tie_qk_r2"      # 这一轮的名字，会成为 experiments/msn_skullfix/<name>/
FROM_RUN    = "tie_qk"         # 照抄这个 run 的全部超参；留空 "" 则用下面的 flags
EXTRA_FLAGS = []               # 例：["--loss", "cd", "--repulsion-weight", "0.5"]
# ===================================================================

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
PY_MSN = "/root/miniconda3/envs/comp0190-msn/bin/python"
OUT_DIR = os.path.join(REPO, "experiments", "msn_skullfix", RUN_NAME)

CMD = [PY_MSN, "src/models/train_skullfix.py", "--run-name", RUN_NAME]
if FROM_RUN:
    CMD += ["--from-run", FROM_RUN]
CMD += EXTRA_FLAGS

print("REPO      ", REPO)
print("输出目录   ", os.path.relpath(OUT_DIR, REPO))
print("\n完整命令（要在终端跑就复制这一行）:")
print("  " + " ".join(CMD))
print("\n可用的 flag: python src/models/train_skullfix.py --help")

## 2. 预检

四件事：数据在不在、配对对齐还对不对、显存是不是空的、`RUN_NAME` 有没有被占用。
**全部只用 CPU/numpy，不碰显存。**

配对对齐自检看的是"每个 GT 点到最近输入点的距离"：中位数应该≈采样间距（共享表面），
只有尾部（缺损区）才该很大。**中位数很大 = 两朵点云不在同一个坐标系**，那是数据 bug，
训练出来的一切都不可信。

In [ ]:
CACHE = os.path.join(REPO, "data", "cache", "skullfix_pairs_4096_6144.npz")
assert os.path.exists(CACHE), f"缓存不存在，先跑附录 A：\n{CACHE}"

data = np.load(CACHE)
ids, inputs, gt = data["ids"], data["inputs"], data["gt"]
scale_mm = float(data["scale_mm"].mean())
print(f"数据   {len(ids)} 对 | input {inputs.shape} | gt {gt.shape} | scale {scale_mm:.1f} mm")


def _nn_dist(query, ref, chunk=1024):
    """逐块纯 numpy 最近邻距离。这个 kernel 故意不装 scipy（版本被 numpy<2 +
    tensorflow<2.16 钉死），一个自检不值得动它。"""
    ref2 = (ref ** 2).sum(1)
    out = np.empty(len(query), dtype=np.float64)
    for i in range(0, len(query), chunk):
        q = query[i:i + chunk]
        d2 = (q ** 2).sum(1)[:, None] - 2.0 * (q @ ref.T) + ref2[None, :]
        out[i:i + chunk] = np.sqrt(np.maximum(d2.min(1), 0.0))
    return out


nn = _nn_dist(gt[0].astype(np.float64), inputs[0].astype(np.float64)) * scale_mm
ok = np.median(nn) < 4.0
print(f"配对对齐 skull_{ids[0]}: 中位 {np.median(nn):.2f} mm（共享表面，应 < 4）"
      f" | p99 {np.percentile(nn, 99):.2f} mm（缺损区，应 > 10）  {'✅' if ok else '❌ 数据有问题'}")

print("\n显存:")
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
      " ← used 应接近 0；不为 0 说明有别的 kernel 占着，训练会 OOM")

taken = [f for f in ("run.json", "best.h5", "history.csv")
         if os.path.exists(os.path.join(OUT_DIR, f))]
print(f"\nrun 名 '{RUN_NAME}': " + ("✅ 可用" if not taken else
      f"❌ 已被占用（{', '.join(taken)}）—— 换个名字，训练脚本会拒绝启动"))

## 3. 训练

单张 4090 上约 **9 s/epoch**（80 训练 + 20 验证）。近期各轮跑了 222~411 轮，
即 **35~60 分钟**。

### 什么时候停？三道闸，实际起作用的是第一道

| 机制 | 值 | 说明 |
|---|---|---|
| `EarlyStopping` | patience 20 | **真正的停止信号**。val_loss 连续 20 轮不改善就停，并恢复最优权重 |
| `ReduceLROnPlateau` | patience 10（由早停推导） | 停滞 10 轮先把 lr 砍半。**必须小于早停 patience，否则永不触发** |
| `--epochs` / `--minutes` | 500 / 150 min | 兜底。⚠️ **撞到 `--epochs` 上限的 run 要作废**（`cd_rep05_truncated` 就是），因为最优值落在最后一轮 = 还在收敛中 |

### 已有各轮的收敛情况（全部在 LR 修复之后，可比）

| run | 配置 | epochs | best val CD_t |
|---|---|---:|---:|
| `lr_fix_only` | CD+DCD | 279 | 6.317 |
| `rep_w05` | CD+DCD+rep | 222 | 6.326 |
| `cd_only` | CD 单独 | 305 | 6.353 |
| `cd_rep05_full` | **CD+rep** | 256 | **6.267** |
| `cd_rep05_r2` | 同上，重复一次 | 249 | 6.274 |
| `notext` | CD+rep，去文本 | 380 | 6.225 |
| `tie_qk` | CD+rep，Q/K 绑定 | 411 | 6.198 |
| `pp_attn` | CD+rep，逐点注意力 ❌ | 355 | 6.581 |

（这一列是 `run.json` 的 `best_val_cd_t_mm` 口径，比 `report.py` 的逐颅骨口径系统性低约 0.09mm，**两者不要混着引用**。）

⚠️ **轮数差别很大（222~411），而报告的是"全程最优"** —— 跑得久的天然占便宜。
跨 run 比较时要看 compare notebook 的**同轮次表**，不要只看这一列。

In [ ]:
proc = subprocess.Popen(CMD, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print("\nreturn code:", proc.returncode)

## 4. 本轮自检

**这一节问的是"这次跑得对不对"，不是"结果好不好"。** 结果好不好去 compare notebook。

四个红灯：

| 检查 | 正常 | 不正常说明什么 |
|---|---|---|
| 停止原因 | `EarlyStopping` | **撞 `--epochs` 上限 → 这一轮作废**，调高上限重跑 |
| LR 衰减次数 | 6~9 次 | **0 次 = 学习率衰减没触发**，这一轮的"最优值"是随机游走上的最小值，不可读 |
| 末 30 轮 std | < 0.02 mm | 大（~0.25mm）= 没退火好，同上 |
| val/train CD_t | 1.0~1.2× | 远大于 1.2 才谈得上过拟合（本模型历来 1.03~1.19×，从没过拟合过） |

In [ ]:
import pandas as pd

meta = json.load(open(os.path.join(OUT_DIR, "run.json")))
hist = pd.read_csv(os.path.join(OUT_DIR, "history.csv"))
best_ep = int(hist["val_loss"].idxmin()) + 1
n_ep = len(hist)
lr_drops = sum(1 for i in range(1, n_ep)
               if hist["lr"][i] < hist["lr"][i - 1] - 1e-12)
late_std = (hist["val_cd_t_metric"] * meta["scale_mm"]).tail(30).std()
ratio = meta["final"]["val_cd_t_metric"] / meta["final"]["cd_t_metric"]

# 先判早停：它是决定性的（最后 patience 轮没有改善）。反过来先判上限会误伤 ——
# 早期的 run.json 没记 --epochs，回退值可能比它实际用的上限小（cd_only 就被误报过）。
if n_ep - best_ep == meta["early_stop_patience"]:
    stop = "✅ EarlyStopping"
elif "epochs" not in meta:
    stop = "⚠️ run.json 没记 --epochs 上限，无法判定是否被截断"
elif n_ep >= meta["epochs"]:
    stop = "❌ 撞 --epochs 上限被截断 —— 这一轮不可引用，调高上限重跑"
else:
    stop = "⚠️ 被 --minutes 墙钟预算掐停"

print(f"run          {RUN_NAME}   ({meta['loss']}, rep={meta['repulsion_weight']:g}, "
      f"tie_qk={meta.get('tie_qk_init', False)}, use_text={meta.get('use_text', True)})")
print(f"停止原因      {stop}")
print(f"轮数          {n_ep}（最优在第 {best_ep} 轮，之后 {n_ep - best_ep} 轮无改善）")
print(f"LR 衰减       {lr_drops} 次      {'✅' if lr_drops >= 5 else '❌ 太少，检查配置'}")
print(f"末 30 轮 std  {late_std:.4f} mm  {'✅ 已退火' if late_std < 0.02 else '❌ 没退火好'}")
print(f"val/train     {ratio:.2f}×       {'✅ 正常' if ratio < 1.25 else '⚠️ 偏高'}")
print(f"\nbest val CD_t {meta['best_val_cd_t_mm']:.3f} mm   ← run.json 口径，"
      f"和 compare notebook 的逐颅骨口径差约 0.09mm，不要混引")

## 5. 存档

`experiments/` 整个在 `.gitignore` 里（单个 `.h5` 是 716 MiB），但 `run.json` + `history.csv`
很小且**必须进 git** —— 权重丢了还能重训，实验记录丢了就再也复现不出对比表了。

跑完之后还要做的（这个 cell 不替你做）：
1. 在 [`experiments_log/README.md`](../experiments_log/README.md) 的表里加一行，写清楚这一轮是什么、有效性如何
2. 在 [`devlog.md`](../devlog.md) 里按日期追加一节
3. `git add experiments_log/ && git commit`

In [ ]:
dst = os.path.join(REPO, "experiments_log", RUN_NAME)
os.makedirs(dst, exist_ok=True)
for f in ("run.json", "history.csv"):
    shutil.copy2(os.path.join(OUT_DIR, f), os.path.join(dst, f))
print(f"已存档 -> experiments_log/{RUN_NAME}/  ({', '.join(os.listdir(dst))})")
print("\n别忘了：experiments_log/README.md 加一行 + devlog.md 追加一节 + git commit")

## 下一步

判读结果去 **[`MSN_compare_runs.ipynb`](MSN_compare_runs.ipynb)** —— 那边有指标词典、
同轮次表、配对检验和判决清单。

看表面质量/mesh/密度图去 **[`MSN_surface_quality.ipynb`](MSN_surface_quality.ipynb)**。

---
---

## 附录 A — 数据准备

**只在要改点数或样本量时才跑。** 当前 `data/cache/skullfix_pairs_4096_6144.npz` 已经是 100 对的缓存。

只用 CPU（`skimage` marching cubes + `fpsample` 最远点采样），不碰 GPU。
`WORKERS=8` 是实测最优：这个任务是**内存带宽密集**而非算力密集，12 比 8 慢、24 比 4 还慢。

In [ ]:
N_SAMPLES, N_DENSE, N_IN, N_OUT, WORKERS = 0, 16384, 4096, 6144, 8

_out = os.path.join(REPO, "data", "cache", f"skullfix_pairs_{N_IN}_{N_OUT}.npz")
_p = subprocess.Popen(
    [PY_MSN, "src/data/prepare_skullfix.py", "--n-samples", str(N_SAMPLES),
     "--n-dense", str(N_DENSE), "--n-in", str(N_IN), "--n-out", str(N_OUT),
     "--workers", str(WORKERS), "--out", _out],
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in _p.stdout:
        print(line, end="")
finally:
    _p.wait()
print("\nreturn code:", _p.returncode)

## 附录 B — 本实现相对原 demo 改了什么


**A. 数据配对错位（正确性 bug）**
`explore_skull.ipynb` 的 `nrrd_to_point_cloud` 对 complete 和 defective **各自独立**做
`normalize_point_cloud`。切掉一块骨头会移动质心、改变最大半径，配对的两朵云于是落到
**不同坐标系**。实测 skull_000：质心偏移 7.55 voxel（半径的 3.6%）、尺度差 2.8%，
同点数下 GT→input 最近距离被抬高 32%。现在从 defective 推出**唯一一个**相似变换同时作用于两者
（推理时只有 defective 可用，所以这个坐标系可复现）。

**B. 距离计算把显存吃爆**
demo 的 `distance_matrix` 把两朵云 tile 成 `(B,N,M,3)`。按 demo 自己的设置（batch 8、6144 点）
光这一个张量前向就 ~12 GB，反向还要留着——这就是推理 notebook 当初被迫退回 CPU 的原因。
换成 `|a|²-2a·b+|b|²` 只产生 `(B,N,M)`。**改完之后完整论文架构（187.5M 参数）在单张 4090 上
372 ms/步、15.5 GiB，训得动了**，不需要缩模型。

**C. DCD 无法从随机初始化引导**
DCD 有界于 [0,2]，两个因子在预测偏离时同时消失。本模型初始化实测：最近邻距离均值 2.79 →
`exp(-2.79)=0.067`；6144 个 GT 点全部坍缩到 **6 个**不同预测点 → 密度权重 ~1/1970。
两者相乘让 loss 钉在 1.9995（上界 2.0），在 1e-7/1e-4/3e-4/1e-3 上扫 40 步 loss 变化不到 0.03。
默认改用 `cd_dcd`：CD 无界负责把形状拉对，DCD 在形状对上后接管精修。

**D. 推理不确定性**
demo 的 `UniformSampler` 用有状态随机数抽质心，**同一个模型对同一输入两次调用输出差 1.03**。
现在训练时保持随机（对 40 个样本相当于免费增广），推理时改用固定种子的 stateless 抽样，
重复调用逐位一致——指标才可复现。

**E. 其它**
- 学习率 1e-7 → 3e-4 + 100 步 warmup。1e-7 配 Adam 比常规小三个数量级。
- batch 8 → 4（24 GB 上 8 会 OOM）。模型里**没有 BatchNorm**（`LBR` 只是 Dense+ReLU），小 batch 只增加梯度噪声。
- `validation_split=0.1` → 按颅骨 id 显式划分。Keras 是**先切尾部再打乱**，一旦每颗颅骨生成多个 partial
  （demo 的 `preprocess_data` 就生成 2 个），兄弟样本会横跨划分边界造成泄漏。
- 冻结的 BERT 预计算。`trainable=False` 且只有 "skull" 一个类别 → 输出是常量，每步重算 1.1 亿参数是浪费。
- checkpoint 存成 `best.h5` 而非 `best.weights.h5`。后者是新版 Keras 格式，即使
  `save_weights_only=True` 也会把 Adam 的动量一起存（187.5M → 562M 个值，**2.25 GB**），
  且每次 val 改善都重写一遍。旧格式只存权重（750 MB）。
- 体素间距。nrrd 头里是各向异性带剪切的 `space directions`（0.451/0.446/0.625 mm），
  demo 在索引空间做 marching cubes，颅骨沿 z 被拉伸约 39%。现在应用该变换并存下 `scale_mm`，
  指标可直接换算成毫米。